# Plotting diagnostics playground

In [2]:
# modules needed
import sys 
import os
import pandas as pd
import numpy as np
import plotly.express as px
sys.path.append(os.path.abspath("/Users/hkershaw/DART/Projects/Diagnostics/pyDART/src/obs_sequence"))
import obs_sequence as dart_os

/var/folders/qv/qpqwspdx2mn4zndqj91cxnkwzfytzl/T/ipykernel_75677/3161982150.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:
# defne functions used

def select_by_dart_qc(df, dart_qc):
    """
    Selects rows from a DataFrame based on the DART quality control flag.

    Parameters:
    df (DataFrame): A pandas DataFrame.
    dart_qc (int): The DART quality control flag to select.

    Returns:
    DataFrame: A DataFrame containing only the rows with the specified DART quality control flag.

    Raises:
    ValueError: If the DART quality control flag is not present in the DataFrame.
    """
    if dart_qc not in df['DART_quality_control'].unique():
        raise ValueError(f"DART quality control flag '{dart_qc}' not found in DataFrame.")
    else:
        return df[df['DART_quality_control'] == dart_qc]

def mean_then_sqrt(x):
    """
    Calculates the mean of an array-like object and then takes the square root of the result.

    Parameters:
    arr (array-like): An array-like object (such as a list or a pandas Series). 
                      The elements should be numeric.

    Returns:
    float: The square root of the mean of the input array.

    Raises:
    TypeError: If the input is not an array-like object containing numeric values.
    """
    return np.sqrt(np.mean(x))

def rmse_bias_by_obs_type(df, obs_type):
    """
    Calculate the RMSE and bias for a given observation type.

    Parameters:
    df (DataFrame): A pandas DataFrame.
    obs_type (str): The observation type for which to calculate the RMSE and bias.

    Returns:
    DataFrame: A DataFrame containing the RMSE and bias for the given observation type.

    Raises:
    ValueError: If the observation type is not present in the DataFrame.
    """
    if obs_type not in df['type'].unique():
        raise ValueError(f"Observation type '{obs_type}' not found in DataFrame.")
    else:
        obs_type_df = df[df['type'] == obs_type]
        obs_type_agg = obs_type_df.groupby('plevels').agg({'sq_err':mean_then_sqrt, 'bias':'mean'}).reset_index()
        obs_type_agg.rename(columns={'sq_err':'rmse'}, inplace=True)
        return obs_type_agg
    
def bin_by_levels(df, levels):
    """
    Bins a DataFrame by vertical level.

    Parameters:
    df (DataFrame): A pandas DataFrame.
    levels (array-like): An array-like object containing the vertical levels to use for binning.

    Returns:
    DataFrame: A DataFrame with an additional column containing the bin labels.

    Raises:
    ValueError: If the binning levels are not present in the DataFrame.
    """
    if df['vertical'].isnull().values.any(): # what about horizontal observations?
        raise ValueError("Missing values in 'vertical' column.")
    else:
        df.loc[:,'plevels'] = pd.cut(df['vertical'], levels)
        return df

In [4]:
# load an obsequence file into a dataframe
obs_seq = dart_os.obs_sequence('obs_seq.final.ascii')
obs_seq.df.head()

,obs_num,observation,prior_ensemble_mean,prior_ensemble_spread,prior_ensemble_member_1,prior_ensemble_member_2,prior_ensemble_member_3,prior_ensemble_member_4,prior_ensemble_member_5,prior_ensemble_member_6,...,latitude,vertical,vert_unit,type,seconds,days,time,obs_err_var,bias,sq_err
0,1,230.16,231.310652,0.405191,231.304725,231.562874,231.333915,231.297690,232.081416,231.051063,...,40.010,23950.0,pressure (Pa),ACARS_TEMPERATURE,75603,153005,2019-12-01 21:00:03,1.00000000000000,1.150652,1.324001
1,2,18.40,15.720527,0.630827,14.217207,15.558196,15.805599,16.594644,14.877743,16.334438,...,40.010,23950.0,pressure (Pa),ACARS_U_WIND_COMPONENT,75603,153005,2019-12-01 21:00:03,6.25000000000000,-2.679473,7.179578
2,3,1.60,-4.932073,0.825899,-5.270562,-5.955998,-4.209766,-5.105016,-4.669405,-4.365305,...,40.010,23950.0,pressure (Pa),ACARS_V_WIND_COMPONENT,75603,153005,2019-12-01 21:00:03,6.25000000000000,-6.532073,42.667980
3,4,264.16,264.060532,0.035584,264.107192,264.097270,264.073212,264.047718,264.074140,264.019895,...,34.105,56260.0,pressure (Pa),ACARS_TEMPERATURE,75603,153005,2019-12-01 21:00:03,1.00000000000000,-0.099468,0.009894
4,5,11.60,10.134115,0.063183,10.067956,10.078798,10.120263,10.084885,10.135112,10.140610,...,34.105,56260.0,pressure (Pa),ACARS_U_WIND_COMPONENT,75603,153005,2019-12-01 21:00:03,6.25000000000000,-1.465885,2.148818


In [5]:
# select only observations with a dart quality control flag of 0
#qc_zero = obs_seq.df[obs_seq.df['DART_quality_control']<=0]
qc_zero = select_by_dart_qc(obs_seq.df, 0)


In [6]:
# bin by vertical level
hPalevels = [0.0, 100.0,  150.0, 200.0, 250.0, 300.0, 400.0, 500.0, 700, 850, 925, 1000, np.inf] # hPa
plevels = [i * 100 for i in hPalevels]
#qc_zero.loc[:,'plevels'] = pd.cut(qc_zero['vertical'], plevels)#
bin_by_levels(qc_zero, plevels)


/var/folders/qv/qpqwspdx2mn4zndqj91cxnkwzfytzl/T/ipykernel_75677/3935188072.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,'plevels'] = pd.cut(df['vertical'], levels)


,obs_num,observation,prior_ensemble_mean,prior_ensemble_spread,prior_ensemble_member_1,prior_ensemble_member_2,prior_ensemble_member_3,prior_ensemble_member_4,prior_ensemble_member_5,prior_ensemble_member_6,...,vertical,vert_unit,type,seconds,days,time,obs_err_var,bias,sq_err,plevels
0,1,230.16,231.310652,0.405191,231.304725,231.562874,231.333915,231.297690,232.081416,231.051063,...,23950.0,pressure (Pa),ACARS_TEMPERATURE,75603,153005,2019-12-01 21:00:03,1.00000000000000,1.150652,1.324001,"(20000.0, 25000.0]"
1,2,18.40,15.720527,0.630827,14.217207,15.558196,15.805599,16.594644,14.877743,16.334438,...,23950.0,pressure (Pa),ACARS_U_WIND_COMPONENT,75603,153005,2019-12-01 21:00:03,6.25000000000000,-2.679473,7.179578,"(20000.0, 25000.0]"
2,3,1.60,-4.932073,0.825899,-5.270562,-5.955998,-4.209766,-5.105016,-4.669405,-4.365305,...,23950.0,pressure (Pa),ACARS_V_WIND_COMPONENT,75603,153005,2019-12-01 21:00:03,6.25000000000000,-6.532073,42.667980,"(20000.0, 25000.0]"
3,4,264.16,264.060532,0.035584,264.107192,264.097270,264.073212,264.047718,264.074140,264.019895,...,56260.0,pressure (Pa),ACARS_TEMPERATURE,75603,153005,2019-12-01 21:00:03,1.00000000000000,-0.099468,0.009894,"(50000.0, 70000.0]"
4,5,11.60,10.134115,0.063183,10.067956,10.078798,10.120263,10.084885,10.135112,10.140610,...,56260.0,pressure (Pa),ACARS_U_WIND_COMPONENT,75603,153005,2019-12-01 21:00:03,6.25000000000000,-1.465885,2.148818,"(50000.0, 70000.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082356,1082357,15.10,18.092244,0.396487,18.172104,18.032478,18.862275,18.022713,18.116851,18.278321,...,69370.0,pressure (Pa),AIRCRAFT_U_WIND_COMPONENT,10800,153006,2019-12-02 03:00:00,9.00000000000000,2.992244,8.953526,"(50000.0, 70000.0]"
1082357,1082358,25.10,18.307062,0.409834,18.431902,18.718723,17.654664,18.481879,18.204611,17.807877,...,69370.0,pressure (Pa),AIRCRAFT_V_WIND_COMPONENT,10800,153006,2019-12-02 03:00:00,9.00000000000000,-6.792938,46.144007,"(50000.0, 70000.0]"
1082370,1082371,234.16,234.262385,0.326980,233.969999,234.425291,234.541376,233.966349,234.269890,235.434338,...,28740.0,pressure (Pa),AIRCRAFT_TEMPERATURE,10800,153006,2019-12-02 03:00:00,1.00000000000000,0.102385,0.010483,"(25000.0, 30000.0]"
1082371,1082372,58.80,57.542819,0.865475,56.099422,56.450182,58.786521,58.000539,57.831022,56.836677,...,28740.0,pressure (Pa),AIRCRAFT_U_WIND_COMPONENT,10800,153006,2019-12-02 03:00:00,9.00000000000000,-1.257181,1.580505,"(25000.0, 30000.0]"


In [8]:
# check for null values, observations that were not binned
qc_zero.isnull().values.any()
#qc_zero[qc_zero['plevels'].isnull()]

False

In [9]:
# calculate the RMSE and bias for the an observation type
#obs_seq_agg = qc_zero.groupby('plevels').agg({'sq_err':mean_then_sqrt, 'bias':'mean'}).reset_index()
#obs_seq_agg.rename(columns={'sq_err':'rmse'}, inplace=True)
mean_bias = rmse_bias_by_obs_type(qc_zero, 'SAT_U_WIND_COMPONENT')
mean_bias

/var/folders/qv/qpqwspdx2mn4zndqj91cxnkwzfytzl/T/ipykernel_75677/3935188072.py:56: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs_type_agg = obs_type_df.groupby('plevels').agg({'sq_err':mean_then_sqrt, 'bias':'mean'}).reset_index()


,plevels,rmse,bias
0,"(0.0, 10000.0]",NaN,NaN
1,"(10000.0, 15000.0]",2.592732,-0.270074
2,"(15000.0, 20000.0]",2.710026,-0.304083
3,"(20000.0, 25000.0]",2.648813,-0.085642
4,"(25000.0, 30000.0]",2.693749,-0.077430
5,"(30000.0, 40000.0]",2.635860,0.032942
6,"(40000.0, 50000.0]",2.355861,0.075882
7,"(50000.0, 70000.0]",1.706947,0.142697
8,"(70000.0, 85000.0]",1.371117,0.106904
9,"(85000.0, 92500.0]",1.185226,-0.127698


In [11]:
#qc_zero['type'].unique()
obs_seq.df['type'].unique()

array(['ACARS_TEMPERATURE', 'ACARS_U_WIND_COMPONENT',
       'ACARS_V_WIND_COMPONENT', 'AIRCRAFT_TEMPERATURE',
       'AIRCRAFT_U_WIND_COMPONENT', 'AIRCRAFT_V_WIND_COMPONENT',
       'GPSRO_REFRACTIVITY', 'AIRS_TEMPERATURE', 'AIRS_SPECIFIC_HUMIDITY',
       'MARINE_SFC_TEMPERATURE', 'MARINE_SFC_SPECIFIC_HUMIDITY',
       'MARINE_SFC_ALTIMETER', 'MARINE_SFC_U_WIND_COMPONENT',
       'MARINE_SFC_V_WIND_COMPONENT', 'LAND_SFC_ALTIMETER',
       'RADIOSONDE_TEMPERATURE', 'RADIOSONDE_SPECIFIC_HUMIDITY',
       'RADIOSONDE_SURFACE_ALTIMETER', 'RADIOSONDE_U_WIND_COMPONENT',
       'RADIOSONDE_V_WIND_COMPONENT', 'SAT_U_WIND_COMPONENT',
       'SAT_V_WIND_COMPONENT'], dtype=object)